In [ ]:
import matplotlib
matplotlib.use("Agg")  # Use non-interactive backend for plots
from pathlib import Path
import pandas as pd
import yaml
from IPython.display import Image, display
from risk_validation.core.metrics.impl import pd as pd_metrics  # noqa: F401
from risk_validation.core.services.pd.metrics_service import PDMetricsService
from risk_validation.core.utils.plots import plot_roc, plot_gini, plot_ks_cdf_with_maxgap
from risk_validation.core.utils.rag import rag_for_metric

DATA_DIR = Path('..') / 'data'

# Load data
csv_path = DATA_DIR / 'sample.csv'
df = pd.read_csv(csv_path)

y_col = 'default_flag'
p_col = 'pred_br'
score_col = 'score_pd'
period_col = 'QTR'

# Split DataFrame into development and validation sets
dev_year = 2019
val_year = 2020

df_dev = df[df['score_year'] == dev_year]
df_val = df[df['score_year'] == val_year]

# Set up PDMetricsService with desired metrics
metric_ids = ['auc_roc', 'gini', 'ks']
service = PDMetricsService(metric_ids)

params = {
    'y_col': y_col,
    'p_col': p_col,
    'score_col': score_col,
}

# Compute metrics for development and validation data
dev_result = service.compute(df_dev, params)
val_result = service.compute(df_val, params)

# Load RAG policy
rag_path = Path('../configs/validation_rag.yaml')
with open(rag_path) as f:
    rag_config = yaml.safe_load(f)
pd_policy = rag_config['pd']

# Calculate RAG status for each metric
for metric_id in metric_ids:
    dev_value = dev_result[dev_result['metric_id'] == metric_id]['value'].iloc[0]
    val_value = val_result[val_result['metric_id'] == metric_id]['value'].iloc[0]
    rag_result = rag_for_metric(
        metric_id=metric_id,
        cfg=pd_policy,
        dev_value=dev_value,
        val_value=val_value
    )
    print(f"{metric_id.upper()}: DEV={dev_value}, VAL={val_value} → RAG: {rag_result['status']}")

# Plot and display metrics
plot_dir = Path('plots')
plot_dir.mkdir(exist_ok=True)

# Generate ROC plot
roc_path = plot_roc(
    y_true=df_val[y_col],
    y_score=df_val[p_col],
    title='ROC Curve - Validation Data',
    out_path=plot_dir / 'roc.png'
)
display(Image(filename=str(roc_path)))

# Generate Gini plot
# Note that plot_gini simply reuses the same roc/gains path logic for its visualization
plot_gini_path = plot_gini(
    y_true=df_val[y_col],
    y_score=df_val[p_col],
    title='Gini Curve - Validation Data',
    out_path=plot_dir / 'gini.png'
)
display(Image(filename=str(plot_gini_path)))

# Generate KS plot
ks_plot_path, _ = plot_ks_cdf_with_maxgap(
    y_true=df_val[y_col],
    y_score=df_val[p_col],
    title='KS CDF - Validation Data',
    out_path=plot_dir / 'ks_cdf.png'
)
display(Image(filename=str(ks_plot_path)))